In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window as W

In [0]:
SALES_STATUS_MAPPING = {
    1: "In process",
    2: "Approved",
    3: "Backordered",
    4: "Rejected",
    5: "Shipped"
}

PURCHASE_STATUS_MAPPING = {
    1: "Pending",
    2: "Approved",
    3: "Rejected",
    4: "Complete"
}


In [0]:
PRODUCT_GOLD_COLUMNS = [
    "product_id",
    "name",
    "days_to_manufacture",
    "product_subcategory",
    "product_category"
]

SALES_ORDER_COLUMNS = [
    "status",
    "product_id",
    "order_qty",
    "unit_price",
    "unit_price_discount",
    "order_date"
]

PURCHASE_ORDER_COLUMNS = [
    "order_qty",
    "product_id",
    "unit_price",
    "status",
    "order_date",
]

In [0]:
CATALOG_SILVER = 'mini_project'
SCHEMA_SILVER = 'silver_layer'

CATALOG_GOLD = 'mini_project'
SCHEMA_GOLD = 'gold_layer'

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_GOLD}.{SCHEMA_GOLD}")

In [0]:
product_df = spark.read.table(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.product")
product_subcategory_df = spark.read.table(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.product_subcategory")
product_category_df = spark.read.table(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.product_category")

In [0]:
p  = product_df.alias("p")
psc = product_subcategory_df.alias("psc")
pc  = product_category_df.alias("pc")

product_df = (
    p
    .join(
        psc.select(
            F.col("product_subcategory_id"),
            F.col("product_category_id"),
            F.col("name").alias("product_subcategory")
        ),
        on="product_subcategory_id",
        how="left"
    )
    .join(
        pc.select(
            F.col("product_category_id"),
            F.col("name").alias("product_category")
        ),
        on="product_category_id",
        how="left"
    )
)

In [0]:
(
    product_df.select([
        F.col(i) for i in PRODUCT_GOLD_COLUMNS
    ]).write
    .mode("overwrite").saveAsTable(f"{CATALOG_GOLD}.{SCHEMA_GOLD}.product")
)

In [0]:
sales_order_detail_df = spark.read.table(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.sales_order_detail")
sales_order_header_df = spark.read.table(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.sales_order_header")

In [0]:
def with_casted_status(df: DataFrame, status: dict):
    mapping_expr = F.create_map(
    *[F.lit(x) for x in sum(status.items(), ())]
    )

    return df.withColumn(
        "status",
        mapping_expr[F.col("status")]
    )

In [0]:
sod = sales_order_detail_df.alias("sod")
soh = sales_order_header_df.alias("soh")
(
    sod
    .join(
        soh.select(
            F.col("status"),
            F.col("order_date"),
            F.col("sales_order_id")
        ),
        on="sales_order_id",
        how="inner"
    ).select(SALES_ORDER_COLUMNS)
    .transform(lambda x: with_casted_status(x, SALES_STATUS_MAPPING))
    .write.mode("overwrite")
    .saveAsTable(f"{CATALOG_GOLD}.{SCHEMA_GOLD}.sales_order")
)

In [0]:
purchase_order_detail_df = spark.read.table(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.purchase_order_detail")
purchase_order_header_df = spark.read.table(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.purchase_order_header")

In [0]:
pod  = purchase_order_detail_df.alias("pod")
poh = purchase_order_header_df.alias("poh")
(
    pod
    .join(
        poh.select(
            F.col("status"),
            F.col("order_date"),
            F.col("purchase_order_id")
        ),
        on="purchase_order_id",
        how="inner"
    ).select(PURCHASE_ORDER_COLUMNS)
    .transform(lambda x: with_casted_status(x, PURCHASE_STATUS_MAPPING))
    .write.mode("overwrite")
    .saveAsTable(f"{CATALOG_GOLD}.{SCHEMA_GOLD}.purchase_order")
)

In [0]:
from pyspark.sql import functions as F

# 1) Get min and max dates across both header tables
p_minmax = purchase_order_header_df.select(
    F.min(F.to_date("order_date")).alias("min_date"),
    F.max(F.to_date("order_date")).alias("max_date")
)

s_minmax = sales_order_header_df.select(
    F.min(F.to_date("order_date")).alias("min_date"),
    F.max(F.to_date("order_date")).alias("max_date")
)

bounds = p_minmax.unionByName(s_minmax).agg(
    F.min("min_date").alias("min_date"),
    F.max("max_date").alias("max_date")
).collect()[0]

min_date = bounds["min_date"]
max_date = bounds["max_date"]

# 2) Create a contiguous list of dates using spark.range
#    num_days = datediff(max_date, min_date) + 1
num_days = purchase_order_header_df.sparkSession.createDataFrame([(min_date, max_date)], ["min_date","max_date"]) \
    .select((F.datediff("max_date", "min_date") + F.lit(1)).alias("num_days")) \
    .collect()[0]["num_days"]

date_dim_df = (
    purchase_order_header_df.sparkSession
    .range(0, int(num_days))
    .select(F.date_add(F.lit(min_date), F.col("id").cast("int")).alias("date"))
    .drop("id")
)

# 3) Add standard date dimension attributes
dim_date_df = (
    date_dim_df
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))  # common surrogate key
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_of_week", F.dayofweek("date"))  # 1=Sunday..7=Saturday
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("is_weekend", F.col("day_of_week").isin([1, 7]))
    .withColumn("is_month_start", F.col("date") == F.trunc(F.col("date"), "MM"))
    .withColumn("is_month_end", F.col("date") == F.last_day(F.col("date")))
    .withColumn("is_quarter_start", F.col("date") == F.trunc(F.col("date"), "Q"))
    .withColumn("is_quarter_end", F.col("date") == F.add_months(F.trunc(F.col("date"), "Q"), 3) - F.expr("INTERVAL 1 DAY"))
    .withColumn("is_year_start", F.col("date") == F.trunc(F.col("date"), "YYYY"))
    .withColumn("is_year_end", F.col("date") == (F.add_months(F.trunc(F.col("date"), "YYYY"), 12) - F.expr("INTERVAL 1 DAY")))
).write.mode("overwrite").saveAsTable(f"{CATALOG_GOLD}.{SCHEMA_GOLD}.date_dim")
